In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder


print("Loading raw data...")
brokers = pd.read_csv("synthetic_brokers_v80.csv")
leads = pd.read_csv("synthetic_leads_v80.csv")
assignments = pd.read_csv("synthetic_assignments_v80.csv")
counterfactual = pd.read_csv("synthetic_counterfactual_v80.csv")
historical = pd.read_csv("synthetic_historical_v80.csv")

Loading raw data...


/tmp/ipykernel_70112/436716056.py:11: DtypeWarning: Columns (33,35) have mixed types. Specify dtype option on import or set low_memory=False.
  historical = pd.read_csv("synthetic_historical_v80.csv")


In [3]:
reentry_mask = assignments["lead_id"].str.contains("_R", na=False)
reentry_lead_ids = assignments.loc[reentry_mask, "lead_id"].unique()
print(f"Re-entry lead IDs in assignments: {len(reentry_lead_ids):,}")

# Recover re‑entry lead rows from historical
hist_cols_for_leads = [
    "lead_id", "lead_date", "region", "insurance_type", "language",
    "tenure_years", "digital_engagement_score", "quote_value",
    "lead_difficulty", "sophistication", "patience_hours",
    "claims_severity", "multi_product_intent", "hour_of_day",
    "is_weekend", "month", "postal_code_prefix",
]
hist_cols_for_leads = [c for c in hist_cols_for_leads if c in historical.columns]

reentry_leads_recovered = (
    historical.loc[historical["lead_id"].isin(reentry_lead_ids), hist_cols_for_leads]
    .drop_duplicates(subset=["lead_id"])
)
reentry_leads_recovered = reentry_leads_recovered.copy()
reentry_leads_recovered["original_lead_id"] = (
    reentry_leads_recovered["lead_id"].str.replace(r"_R\d+$", "", regex=True)
)

leads_full = pd.concat([leads, reentry_leads_recovered], ignore_index=True)
if "original_lead_id" not in leads_full.columns:
    leads_full["original_lead_id"] = leads_full["lead_id"]
else:
    leads_full["original_lead_id"] = leads_full["original_lead_id"].fillna(leads_full["lead_id"])

print(f"Leads after re-entry fix: {len(leads_full):,}")

Re-entry lead IDs in assignments: 850
Leads after re-entry fix: 11,474


In [4]:
known_broker_ids = set(brokers["broker_id"])
used_broker_ids = set(assignments["broker_id"].dropna())
orphan_broker_ids = used_broker_ids - known_broker_ids
print(f"Orphaned broker IDs: {len(orphan_broker_ids):,}")

hist_cols_for_brokers = [
    "broker_id", "region", "expertise_auto", "expertise_home",
    "expertise_bundle", "conversion_rate", "csat_score", "languages",
    "ribo_licensed", "ribo_license_years", "capacity", "avg_response_time",
    "is_new_broker", "skill_level", "reliability", "commission_rate",
    "cost_per_lead", "efficiency", "burnout_risk",
]
hist_cols_for_brokers = [c for c in hist_cols_for_brokers if c in historical.columns]

replacement_brokers_recovered = (
    historical.loc[historical["broker_id"].isin(orphan_broker_ids), hist_cols_for_brokers]
    .drop_duplicates(subset=["broker_id"])
)
for col in ["years_experience", "current_caseload"]:
    if col not in replacement_brokers_recovered.columns:
        replacement_brokers_recovered[col] = np.nan
replacement_brokers_recovered["is_new_broker"] = (
    replacement_brokers_recovered.get("is_new_broker", pd.Series(True))
    .fillna(True)
)

brokers_full = pd.concat([brokers, replacement_brokers_recovered], ignore_index=True)
print(f"Brokers after replacement fix: {len(brokers_full):,}")

Orphaned broker IDs: 6
Brokers after replacement fix: 306


/tmp/ipykernel_70112/3321381941.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(True)


In [5]:
actual_caseload = (
    assignments[assignments["is_assigned"] == 1]
    .groupby("broker_id")
    .size()
    .rename("assignment_count")
    .reset_index()
)
brokers_full = brokers_full.merge(actual_caseload, on="broker_id", how="left")
brokers_full["assignment_count"] = brokers_full["assignment_count"].fillna(0)
brokers_full["utilization"] = np.clip(
    brokers_full["assignment_count"] / brokers_full["capacity"], 0, 3
).round(4)
brokers_full["is_overloaded"] = (brokers_full["assignment_count"] > brokers_full["capacity"]).astype(int)


In [6]:
valid_lead_ids = set(leads_full["lead_id"])
valid_broker_ids = set(brokers_full["broker_id"])

assignments_clean = assignments[
    assignments["lead_id"].isin(valid_lead_ids) & assignments["broker_id"].isin(valid_broker_ids)
].copy()
print(f"Assignments after referential fix: {len(assignments_clean):,}")


Assignments after referential fix: 54,136


In [7]:
counterfactual_clean = counterfactual[
    counterfactual["lead_id"].isin(valid_lead_ids) & counterfactual["broker_id"].isin(valid_broker_ids)
].copy()
print(f"Counterfactual after fix: {len(counterfactual_clean):,}")

broker_cols_to_merge = [
    "broker_id", "region", "expertise_auto", "expertise_home",
    "expertise_bundle", "conversion_rate", "csat_score", "languages",
    "ribo_licensed", "ribo_license_years", "capacity", "avg_response_time",
    "is_new_broker", "skill_level", "reliability", "commission_rate",
    "cost_per_lead", "efficiency", "burnout_risk",
    "utilization", "is_overloaded",
]
broker_cols_to_merge = [c for c in broker_cols_to_merge if c in brokers_full.columns]

historical_fixed = (
    assignments_clean
    .merge(leads_full, on="lead_id", how="inner")
    .merge(brokers_full[broker_cols_to_merge], on="broker_id", how="inner")
)

for col in ["_score", "_simulated_score"]:
    if col in historical_fixed.columns:
        historical_fixed.drop(columns=[col], inplace=True)

df = historical_fixed.copy()
print(f"Historical merged shape: {df.shape}")

Counterfactual after fix: 34,422
Historical merged shape: (54136, 58)


In [8]:
informative_missing_cols = ["insurance_type", "language", "tenure_years", "digital_engagement_score"]
for col in informative_missing_cols:
    if col in df.columns:
        flag_col = f"{col}_missing"
        df[flag_col] = df[col].isna().astype(int)
        print(f"Created {flag_col} (missing rate: {df[col].isna().mean()*100:.1f}%)")

Created insurance_type_missing (missing rate: 9.4%)
Created language_missing (missing rate: 9.4%)
Created tenure_years_missing (missing rate: 8.6%)
Created digital_engagement_score_missing (missing rate: 7.9%)


In [9]:
categorical_impute = {
    "insurance_type": "UNKNOWN",
    "language": "UNKNOWN",
    "claims_severity": "none",
}
for col, fill_val in categorical_impute.items():
    if col in df.columns:
        before = df[col].isna().sum()
        df[col] = df[col].fillna(fill_val)
        print(f"Imputed {col}: {before:,} NaN → '{fill_val}'")


Imputed insurance_type: 5,102 NaN → 'UNKNOWN'
Imputed language: 5,071 NaN → 'UNKNOWN'
Imputed claims_severity: 3,993 NaN → 'none'


In [10]:
numeric_impute_cols = ["tenure_years", "digital_engagement_score", "lead_difficulty", "sophistication", "patience_hours"]
for col in numeric_impute_cols:
    if col in df.columns:
        before = df[col].isna().sum()
        med_val = df[col].median()
        df[col] = df[col].fillna(med_val)
        print(f"Imputed {col}: {before:,} NaN → median={med_val:.3f}")


Imputed tenure_years: 4,648 NaN → median=1.000
Imputed digital_engagement_score: 4,296 NaN → median=34.200
Imputed lead_difficulty: 3,993 NaN → median=1.002
Imputed sophistication: 3,993 NaN → median=0.510
Imputed patience_hours: 3,993 NaN → median=39.800


In [11]:
structural_nan_cols = ["conversion_delay_days", "response_time_hours", "responded"]
for col in structural_nan_cols:
    if col in df.columns:
        print(f"{col} missing: {df[col].isna().mean()*100:.1f}% (structural)")

conversion_delay_days missing: 95.6% (structural)
response_time_hours missing: 68.8% (structural)
responded missing: 62.4% (structural)


In [12]:
if "multi_product_intent" in df.columns:
    df["multi_product_intent"] = df["multi_product_intent"].fillna(0).astype(int)

if "revenue" in df.columns and "cost" in df.columns:
    df["net_margin"] = np.where(
        df["revenue"] > 0,
        (df["revenue"] - df["cost"]) / df["revenue"].replace(0, np.nan),
        np.nan,
    )
    df["net_margin"] = df["net_margin"].fillna(0).round(4)

if "quote_value" in df.columns and "cost" in df.columns:
    df["expected_roi"] = (
        (df["quote_value"] * df.get("commission_rate", 0.10) - df["cost"])
        / df["cost"].replace(0, np.nan)
    ).round(4)
    df["expected_roi"] = df["expected_roi"].fillna(0)

In [13]:
if "response_time_hours" in df.columns:
    df["response_time_bucket"] = pd.cut(
        df["response_time_hours"],
        bins=[-np.inf, 2, 6, 12, 24, 48, np.inf],
        labels=["<2h", "2-6h", "6-12h", "12-24h", "24-48h", ">48h"],
    )
    rt_order = {"<2h": 0, "2-6h": 1, "6-12h": 2, "12-24h": 3, "24-48h": 4, ">48h": 5}
    df["response_time_bucket_ord"] = (
        df["response_time_bucket"].astype(str).map(rt_order).fillna(-1).astype(int)
    )
    print("Created response_time_bucket_ord")

Created response_time_bucket_ord


In [14]:
if "current_caseload" in df.columns and "capacity" in df.columns:
    df["workload_ratio"] = np.clip(
        df["current_caseload"] / df["capacity"].replace(0, np.nan), 0, 3
    ).fillna(0).round(4)
    print("Created dynamic workload_ratio from current_caseload")
else:
    # Fallback: use utilization (which is based on total assignments)
    if "utilization" in df.columns:
        df["workload_ratio"] = np.clip(df["utilization"], 0, 3).round(4)
        print("Created workload_ratio from utilization (fallback)")

Created workload_ratio from utilization (fallback)


In [15]:
def expertise_match(row):
    ins = row.get("insurance_type", "UNKNOWN")
    if ins == "auto":
        return int(row.get("expertise_auto", 0) == 1)
    elif ins == "home":
        return int(row.get("expertise_home", 0) == 1)
    elif ins == "bundle":
        return int(row.get("expertise_bundle", 0) == 1)
    return 0.5

df["expertise_match"] = df.apply(expertise_match, axis=1)
print("Created expertise_match")

Created expertise_match


In [16]:
def language_match(row):
    lead_lang = row.get("language", "UNKNOWN")
    broker_lang = row.get("languages", "English")
    if broker_lang == "Bilingual":
        return 1.0
    if lead_lang == "UNKNOWN":
        return 0.8
    return 1.0 if lead_lang == broker_lang else 0.3

df["language_match"] = df.apply(language_match, axis=1)
print("Created language_match")

Created language_match


In [17]:
if "lead_date" in df.columns:
    lead_dt = pd.to_datetime(df["lead_date"])
    df["lead_dayofweek"] = lead_dt.dt.dayofweek
    df["lead_weekofyear"] = lead_dt.dt.isocalendar().week.fillna(0).astype(int)
    df["lead_quarter"] = lead_dt.dt.quarter
    df["lead_year"] = lead_dt.dt.year
    print("Created lead temporal features")

Created lead temporal features


In [18]:
for col in ["conversion_rate", "csat_score", "skill_level", "reliability"]:
    if col in df.columns:
        min_val = df[col].min()
        max_val = df[col].max()
        if max_val > min_val:
            df[f"{col}_norm"] = (df[col] - min_val) / (max_val - min_val)
        else:
            df[f"{col}_norm"] = 0.5

broker_quality_cols = [c for c in ["conversion_rate_norm", "csat_score_norm", "skill_level_norm", "reliability_norm"] if c in df.columns]
if broker_quality_cols:
    df["broker_quality_score"] = df[broker_quality_cols].mean(axis=1).round(4)
    print(f"Created broker_quality_score from {broker_quality_cols}")

Created broker_quality_score from ['conversion_rate_norm', 'csat_score_norm', 'skill_level_norm', 'reliability_norm']


In [19]:
if "broker_quality_score" in df.columns and "quote_value" in df.columns:
    df["quality_x_value"] = (df["broker_quality_score"] * df["quote_value"] / df["quote_value"].max()).round(4)
    print("Created quality_x_value")


Created quality_x_value


In [21]:
df.columns

Index(['lead_id', 'original_lead_id_x', 'broker_id', 'assignment_date',
       'converted', 'censored', 'conversion_delay_days', 'conversion_value',
       'revenue', 'cost', 'profit', 'expected_profit', 'is_assigned',
       'interaction_number', 'responded', 'response_time_hours',
       'propensity_score', 'exploration_rate_at_time', 'action_probabilities',
       'position_bias', 'market_regime', 'lead_date', 'region_x',
       'postal_code_prefix', 'insurance_type', 'language', 'tenure_years',
       'digital_engagement_score', 'quote_value', 'lead_difficulty',
       'sophistication', 'patience_hours', 'claims_severity',
       'multi_product_intent', 'hour_of_day', 'is_weekend', 'month',
       'original_lead_id_y', 'region_y', 'expertise_auto', 'expertise_home',
       'expertise_bundle', 'conversion_rate', 'csat_score', 'languages',
       'ribo_licensed', 'ribo_license_years', 'capacity', 'avg_response_time',
       'is_new_broker', 'skill_level', 'reliability', 'commission_r

In [22]:
claims_risk_map = {"none": 0, "minor": 1, "major": 2}

df["claims_risk"] = df["claims_severity"].map(claims_risk_map).fillna(0).astype(int)
print("Created claims_risk ordinal encoding")

Created claims_risk ordinal encoding


In [23]:
df["ribo_x_expertise"] = df["ribo_licensed"] * df["expertise_match"]
df["claims_x_skill"]   = df["claims_risk"] * df["skill_level"]
df["tenure_x_quality"] = df["tenure_years"] * df["broker_quality_score"]
print("Created ribo_x_expertise, claims_x_skill, tenure_x_quality")

Created ribo_x_expertise, claims_x_skill, tenure_x_quality


In [25]:
if "region_x" in df.columns and "region_y" in df.columns:
    df["region_mismatch"] = (df["region_x"] != df["region_y"]).astype(int)
    print("Created region_mismatch")


Created region_mismatch


In [26]:
log_transform_cols = [
    "conversion_delay_days",
    "response_time_hours",
    "quote_value",
    "patience_hours",
]
print("\nLog1p transforms (first):")
for col in log_transform_cols:
    if col in df.columns:
        df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))
        print(f"  Created log_{col}")


Log1p transforms (first):
  Created log_conversion_delay_days
  Created log_response_time_hours
  Created log_quote_value
  Created log_patience_hours


In [27]:
outlier_cap_cols = [
    "log_conversion_delay_days",
    "log_response_time_hours",
    "profit",
    "expected_profit",
    "log_patience_hours",
    "log_quote_value",
]
print("\nOutlier capping at 99th percentile:")
for col in outlier_cap_cols:
    if col in df.columns:
        p99 = df[col].quantile(0.99)
        p01 = df[col].quantile(0.01)
        before_high = (df[col] > p99).sum()
        before_low = (df[col] < p01).sum()
        df[col] = np.clip(df[col], p01, p99)
        print(f"  {col}: capped {before_high} high / {before_low} low values")


Outlier capping at 99th percentile:
  log_conversion_delay_days: capped 23 high / 0 low values
  log_response_time_hours: capped 168 high / 0 low values
  profit: capped 542 high / 408 low values
  expected_profit: capped 542 high / 531 low values
  log_patience_hours: capped 0 high / 0 low values
  log_quote_value: capped 501 high / 499 low values


In [28]:
lang_dummies = pd.get_dummies(df["languages"], prefix="lang", dtype=int)
df = pd.concat([df, lang_dummies], axis=1)
print(f"One-hot encoded 'languages': {lang_dummies.shape[1]} columns")

One-hot encoded 'languages': 3 columns


In [29]:
le_cols = ["insurance_type", "market_regime", "claims_severity"]
for col in le_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[f"{col}_enc"] = le.fit_transform(df[col].astype(str))
        print(f"Label-encoded {col} -> {col}_enc")

Label-encoded insurance_type -> insurance_type_enc
Label-encoded market_regime -> market_regime_enc
Label-encoded claims_severity -> claims_severity_enc


In [30]:
DROP_COLS = [
    "original_lead_id_x", "original_lead_id_y", "postal_code_prefix",
    "action_probabilities", "conversion_value", "current_caseload",
    "assignment_date", "lead_date", "market_regime", "claims_severity",
    "conversion_rate_norm", "csat_score_norm", "skill_level_norm", "reliability_norm",
    "response_time_bucket", "quote_value_tier",
]
DROP_COLS = [c for c in DROP_COLS if c in df.columns]
df_final = df.drop(columns=DROP_COLS)
print(f"Dropped {len(DROP_COLS)} columns: {DROP_COLS}")

Dropped 14 columns: ['original_lead_id_x', 'original_lead_id_y', 'postal_code_prefix', 'action_probabilities', 'conversion_value', 'assignment_date', 'lead_date', 'market_regime', 'claims_severity', 'conversion_rate_norm', 'csat_score_norm', 'skill_level_norm', 'reliability_norm', 'response_time_bucket']


In [31]:
if "responded" in df_final.columns:
    df_final["responded"] = df_final["responded"].fillna(0).astype(int)
    print("Filled responded NaN with 0 (no response)")


Filled responded NaN with 0 (no response)


In [32]:
df_positive = df_final[df_final["is_assigned"] == 1].copy()
df_negative = df_final[df_final["is_assigned"] == 0].copy()
print(f"\nPositive (assigned) samples:  {len(df_positive):,}")
print(f"Negative (unassigned) samples: {len(df_negative):,}")
print(f"Conversion rate (positive):    {df_positive['converted'].mean()*100:.2f}%")


Positive (assigned) samples:  20,354
Negative (unassigned) samples: 33,782
Conversion rate (positive):    11.68%


In [33]:
df_final.to_csv("prepared_full_v81.csv", index=False)
df_positive.to_csv("prepared_positive_v81.csv", index=False)
df_negative.to_csv("prepared_negative_v81.csv", index=False)
leads_full.to_csv("leads_full_v81.csv", index=False)
brokers_full.to_csv("brokers_full_v81.csv", index=False)
counterfactual_clean.to_csv("counterfactual_clean_v81.csv", index=False)

print("\n✓  Data preparation complete. Ready for model training.")
print(f"  prepared_full_v81.csv      — {len(df_final):,} rows × {df_final.shape[1]} cols")
print(f"  prepared_positive_v81.csv  — {len(df_positive):,} rows (assigned only)")
print(f"  prepared_negative_v81.csv  — {len(df_negative):,} rows (unassigned)")
print(f"  leads_full_v81.csv         — {len(leads_full):,} rows (incl. re-entries)")
print(f"  brokers_full_v81.csv       — {len(brokers_full):,} rows (incl. replacements)")
print(f"  counterfactual_clean_v81.csv — {len(counterfactual_clean):,} rows (clean)")


✓  Data preparation complete. Ready for model training.
  prepared_full_v81.csv      — 54,136 rows × 80 cols
  prepared_positive_v81.csv  — 20,354 rows (assigned only)
  prepared_negative_v81.csv  — 33,782 rows (unassigned)
  leads_full_v81.csv         — 11,474 rows (incl. re-entries)
  brokers_full_v81.csv       — 306 rows (incl. replacements)
  counterfactual_clean_v81.csv — 34,422 rows (clean)
